# Getting started: load, explore and export data

This notebook loads data from the Smart Citizen API and from a CSV file, explains blueprints, and exports the data.

Install scdata with the extras you need:

```
pip install scdata              # load, process and export data
pip install "scdata[plot]"      # plots: matplotlib, seaborn, uplot, panel, maps
pip install "scdata[storage]"   # parquet: backups (local or S3) and parquet exports
```

In [ ]:
import logging

import scdata as sc
import smartcitizen_connector
from scdata.tools.custom_logger import set_logger_level

# Only errors: 'WARNING', 'INFO' or 'DEBUG' show what scdata and smartcitizen-connector do
set_logger_level('ERROR')
logging.getLogger('smartcitizen_connector').setLevel('ERROR')

sc.__version__

## Configuration

Blueprints, calibrations and sensor names are read from `BASE_POSTPROCESSING_URL`, the same variable that `smartcitizen-connector` uses. By default it is the [smartcitizen-data](https://github.com/fablabbcn/smartcitizen-data) repository on GitHub. To use the metadata served by Smart Citizen Flows, set it **before importing scdata**:

```
export BASE_POSTPROCESSING_URL=https://flows.smartcitizen.me/api/v1/
```

Other settings (folders, cache) can be changed in `~/.config/scdata/config.yaml`. The file only keeps what differs from the defaults, so the defaults of new versions apply after upgrading.

In [ ]:
from scdata._config import config

print(config._base_postprocessing_url)
print(config.blueprints_urls)
config.paths

## Blueprints

A blueprint describes how to process the data of a kind of device:

- `channels`: calculated channels, with the function that calculates each one
- `checks`: health checks (gaps, flat values, implausible values, outliers)
- `exports`: groups of columns to export
- `meta.kind`: `process` (latest data from the Smart Citizen API, every few hours), `long` (baselines over months of the backups) or `backup`

In [ ]:
list(config.blueprints)

In [ ]:
for name, blueprint in config.blueprints.items():
    print(name, blueprint['meta'].get('kind', 'process'), '-',
          len(blueprint['channels']), 'channels,', len(blueprint['checks']), 'checks,',
          [export['name'] for export in blueprint['exports']], 'exports')

In [ ]:
# The first channels of sc_air
[(channel['name'], channel['function']) for channel in config.blueprints['sc_air']['channels'][:10]]

## A device from the Smart Citizen API

Devices with hardware information in the platform get their blueprint from it, filled with the sensors of their hardware (for example, the calibrations of their electrochemical sensors).

In [ ]:
device = sc.Device(params=sc.APIParams(id=18154))

print(device.blueprint)
print(device.handler.json.name, '- last reading at', device.handler.json.last_reading_at)
print(device.handler.json.postprocessing.hardware_url)

A hardware can have several versions (for example, a sensor replaced on a date). Each version has its own sensors, and `device.process()` processes each period with them:

In [ ]:
[(version.get('from_date'), version.get('to_date'), len(version['channels'])) for version in device.versions]

In [ ]:
# The sensors of the device in the platform
[(sensor.id, sensor.name, sensor.unit) for sensor in device.sensors][:10]

Choose what to load with `device.options`: a period (`min_date`, `max_date`), the `frequency` of the readings, or a `limit` of readings. Here, the last two days:

In [ ]:
import pandas as pd

device.options.min_date = (pd.Timestamp.now(tz='UTC') - pd.Timedelta('2D')).strftime('%Y-%m-%d')
device.options.frequency = '1Min'
device.options

In [ ]:
await device.load()

The data is a `pandas.DataFrame`, with the columns renamed to the names scdata uses and the units converted:

In [ ]:
device.data.head()

In [ ]:
device.data.describe().T.head(10)

A device without hardware information needs a blueprint:

In [ ]:
old_device = sc.Device(blueprint='sc_air', params=sc.APIParams(id=16784))
old_device.options.limit = 500
await old_device.load()
old_device.data.shape

## A device from a CSV file

The CSV has the format of the Smart Citizen API exports (and of the SD card files): the header rows describe the columns. The path is relative to this notebook.

In [ ]:
csv_device = sc.Device(blueprint='sc_air',
                       source={'type': 'csv', 'handler': 'CSVHandler', 'module': 'scdata.io.device_file'},
                       params=sc.CSVParams(id='example', path='../../scdata/tools/interim/example.csv',
                                           timezone='Europe/Madrid'))
await csv_device.load()
csv_device.data.head()

## Tests: groups of devices

A test groups devices to load, process and plot them together. It is stored in `config.paths['processed']`, and the data loaded from the API is cached there (`test.options.cache`), so loading it again only requests new data.

In [ ]:
test = sc.Test(name='EXAMPLES_GETTING_STARTED',
               devices=[sc.Device(params=sc.APIParams(id=device_id)) for device_id in [18154, 17592]],
               force_recreate=True)

for item in test.devices:
    item.options.min_date = device.options.min_date

await test.load()
test.path

In [ ]:
for item in test.devices:
    print(item.id, item.blueprint, item.data.shape)

In [ ]:
test.get_device(17592).data.head()

## Export

`device.export(path)` writes one file per export of the blueprint (`<id>_data_<export>.csv`). `use_exports=False` writes all the columns to `<id>.csv`. CSV (optionally gzipped) and parquet are supported; parquet needs `pip install "scdata[storage]"`.

In [ ]:
from os.path import join

path = join(config.paths['export'], 'examples')
device.export(path, forced_overwrite=True)
device.export(path, forced_overwrite=True, use_exports=False, file_format='parquet')

import os
sorted(os.listdir(path))